In [ ]:
import re
import signal
import unicodedata

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from pymystem3 import Mystem

In [ ]:
mystem = Mystem()

def safe_lemmatize(text):
    try:
        if pd.isna(text) or not isinstance(text, str) or not text.strip():
            return ''
        
        # Обработка SIGPIPE сигнала
        import signal
        signal.signal(signal.SIGPIPE, signal.SIG_DFL)
        
        # Лемматизация с обработкой ошибок
        try:
            lemmas = mystem.lemmatize(text.lower())
            return ''.join(lemmas).strip()
        except Exception as e:
            print(f"Ошибка лемматизации: {e}")
            return ''
            
    except Exception as e:
        print(f"Критическая ошибка: {e}")
        return ''


In [ ]:
def basic_clean(text):
    try:
        # Преобразуем текст в строку и нормализуем Unicode
        text = str(text)
        text = unicodedata.normalize('NFKC', text)
        
        # Очистка HTML-тегов
        text = clean_html(text)
        
        # Удаляем URL
        text = re.sub(r"(https?://\S+)|www\.\S+", " ", text)
        
        # Удаляем email
        text = re.sub(r"\b\w[.-]+?@\w[.-]+\.\w+\b", " ", text, flags=re.IGNORECASE)
        
        # Удаляем телефоны
        text = re.sub(r"(?:\+?\d[\s()-]*){7,}", " ", text)
        
        # Удаляем артикулы
        text = re.sub(r"\ba-zA-Z{2,}\d{2,}|\b\d{5,}\b", " ", text)
        
        # Очищаем от не-буквенных символов
        text = re.sub(r"[^a-zA-Zа-яё0-9\s\-\+%]", " ", text)
        
        # Удаляем лишние пробелы
        text = re.sub(r"\s+", " ", text).strip()
        
        return text
    except Exception as e:
        print(f"Ошибка при очистке текста: {e}")
        return ""

def clean_html(x: str) -> str:
    try:
        return BeautifulSoup(x, "html.parser").get_text(" ")
    except Exception as e:
        print(f"Ошибка при очистке HTML: {e}")
        return str(x)

# Дополнительно добавим проверку на пустые значения
def safe_clean(text):
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return ""
    return basic_clean(text)

In [ ]:
SUSPICIOUS_WORDS = [
    "реплика", "copy", "1:1", "1: 1", "premium copy", "replica", "grade aaa", "aaа", "копия",
    "в стиле", "как у", "под", "аналог", "рефлексия", "турецкая копия", 'без коробки', 'AAA', 'второй сорт',
    "оригинал гарантируем", "подделка", "имитация", "аналог", "fake", "imitation", "analog", "premium copy", "grade aaa",
    "под", "по мотивам", "в духе", "китайская копия", "аналог оригинала", "премиум копия", "люкс копия", "элит копия",
]
SUSPICIOUS_PATTERNS = [re.compile(r"\b" + re.escape(w) + r"\b", re.IGNORECASE) for w in SUSPICIOUS_WORDS]

def count_suspicious(text):
    text = str(text)
    cnt = 0
    for p in SUSPICIOUS_PATTERNS:
        if p.search(text):
            cnt += 1
    return cnt



In [ ]:
def stats_features(text: str, prefix: str) -> pd.Series:
    try:
        text = str(text)
        n_chars = len(text)
        n_digits = sum(ch.isdigit() for ch in text)
        n_upper = sum(ch.isupper() for ch in text)
        n_words = len(text.split())
        avg_word_len = (sum(len(w) for w in text.split()) / (n_words + 1e-6))
        
        # Создаём Series с префиксами
        return pd.Series({
            f"{prefix}_chars": n_chars,
            f"{prefix}_digits": n_digits,
            f"{prefix}_upper": n_upper,
            f"{prefix}_words": n_words,
            f"{prefix}_avg_word_len": avg_word_len
        })
    except Exception as e:
        print(f"Ошибка преобразования: {e}")
        return pd.Series({
            f"{prefix}_chars": 0,
            f"{prefix}_digits": 0,
            f"{prefix}_upper": 0,
            f"{prefix}_words": 0,
            f"{prefix}_avg_word_len": 0
        })

# Модифицируем функцию расчёта статистики
def calculate_stats(df):
    # Обработка текстовых полей с соответствующими префиксами
    for col, prefix in zip(
        ['brand_name', 'description', 'name_rus'],
        ['name', 'desc', 'rus']
    ):
        stats = df[col].astype(str).apply(lambda x: stats_features(x, prefix)).apply(pd.Series)
        df = pd.concat([df, stats], axis=1)
    return df



In [ ]:
def russian_ratio(text):
    if pd.isna(text) or text == '':
        return 0
    text = str(text)
    total_chars = len(text)
    if total_chars == 0:
        return 0  
    russian_letters = len(re.findall(r'[а-яА-ЯёЁ]', text))
    return (russian_letters / total_chars) * 100 if total_chars > 0 else 0


In [ ]:
def safe_returns_divide(a, b):
    result = np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b!=0)
    result = np.where(result == np.inf, np.nan, result)
    result = np.where(result == -np.inf, np.nan, result)
    return result


In [ ]:
def safe_divide(a, b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b!=0)


In [ ]:

messengers = [
 'telegram', 'whatsapp', 'viber', 'icq', 'discord',
 'мессенджер', 'чат', 'vk', 'wechat', 'вк', 'вконтакте', 'vkontakte',
 'телега', 'телеграм', 'телеграмм', 'тг', 'tg',
 'вайбер', 'вотсап', 'ватсап', 'воцап', 'вацап', 'ватсапп', 'вотсапп', 'вацапп',
 'whatsup', 'whatsupp', 'whatsap', 'watsap', 'watsapp'
]

escaped = sorted((re.escape(word) for word in messengers), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(escaped) + r')\b', re.IGNORECASE)

def find_messenger_mentions(text):
    if not isinstance(text, str):
        return  
    return pattern.findall(text)



In [ ]:
def has_messengers(text):
    if isinstance(text, list) and len(text) > 0:
        return 1
    return 0


In [ ]:
def extract_urls(df, column: str) -> pd.DataFrame:
    url_pattern = re.compile(
        r'(https?://[^\s]+|www\.[^\s]+)',  # http, https и www
        flags=re.IGNORECASE
    )
    def find_urls(text):
        urls = url_pattern.findall(str(text))
        return urls if urls else np.nan
    df["urls"] = df[column].apply(find_urls)
    return df


In [ ]:
def has_urls(urls):
    return 1 if isinstance(urls, list) and len(urls) > 0 else 0




In [ ]:
phone_pattern = re.compile(r'(?:\+7|8)[\s\-]?\(?(\d{3})\)?[\s\-]?(\d{3})[\s\-]?(\d{2})[\s\-]?(\d{2})')

def find_phones(text):
    if not isinstance(text, str):
        return []
    matches = phone_pattern.findall(text)
    return [''.join(match) for match in matches]  # например: ['9991234567']



In [ ]:
def has_russian_phones(phones):
    return 1 if len(phones) > 0 else 0



In [ ]:
def clean_returns(x):
    if pd.isna(x):
        return np.nan
    return np.nan if x in [np.inf, -np.inf] else x


In [ ]:
def clean_text_features(text):
    if isinstance(text, list):
        return ' '.join(text)
    return str(text)


In [ ]:
def add_activity_features(df):
    df['photo_to_comment_ratio'] = np.where(
        df['comments_published_count'] != 0,
        df['photos_published_count'] / df['comments_published_count'],
        0
    )
    
    df['video_to_comment_ratio'] = np.where(
        df['comments_published_count'] != 0,
        df['videos_published_count'] / df['comments_published_count'],
        0
    )
    
    # Обработка бесконечностей и NaN
    df['photo_to_comment_ratio'] = df['photo_to_comment_ratio'].replace(
        [np.inf, -np.inf], 0
    ).fillna(0)
    
    df['video_to_comment_ratio'] = df['video_to_comment_ratio'].replace(
        [np.inf, -np.inf], 0
    ).fillna(0)
    
    return df

In [ ]:
def add_time_features(df):
    df['comments_per_time_alive'] = np.where(
        df['item_time_alive'] != 0,
        df['comments_published_count'] / df['item_time_alive'],
        0
    )
    
    df['photos_per_time_alive'] = np.where(
        df['item_time_alive'] != 0,
        df['photos_published_count'] / df['item_time_alive'],
        0
    )
    
    df['video_per_time_alive'] = np.where(
        df['item_time_alive'] != 0,
        df['videos_published_count'] / df['item_time_alive'],
        0
    )
    
    return df


In [ ]:

def add_ratios(df):
    # ratio_7, ratio_30, ratio_90
    for period in [7, 30, 90]:
        col_name = f'ratio_{period}'
        df[col_name] = np.where(
            (df['seller_time_alive'] <= 90) & (df['seller_time_alive'] > 0),
            df[f'ExemplarAcceptedCountTotal{period}'] / df['seller_time_alive'],
            np.nan
        )
    
    # exemplar_to_order_ratio
    for period in [7, 30, 90]:
        col_name = f'exemplar_to_order_ratio_{period}'
        df[col_name] = np.where(
            df[f'OrderAcceptedCountTotal{period}'] > 0,
            df[f'ExemplarAcceptedCountTotal{period}'] / df[f'OrderAcceptedCountTotal{period}'],
            0
        )
    
    # returns_to_orders
    for period in [7, 30, 90]:
        col_name = f'returns_to_orders_{period}'
        df[col_name] = np.where(
            df[f'ExemplarAcceptedCountTotal{period}'] > 0,
            df[f'ExemplarReturnedCountTotal{period}'] / df[f'ExemplarAcceptedCountTotal{period}'],
            0
        )
        df[col_name] = df[col_name].replace([np.inf, -np.inf], np.nan)
    
    # returns_to_time_alive
    for period in [7, 30, 90]:
        col_name = f'returns_to_time_alive_{period}'
        df[col_name] = np.where(
            df['item_time_alive'] > 0,
            df[f'ExemplarReturnedCountTotal{period}'] / df['item_time_alive'],
            0
        )
        df[col_name] = df[col_name].replace([np.inf, -np.inf], np.nan)
    
    return df


In [ ]:
def create_missing_flags(df):
    # Список столбцов для проверки
    columns_to_check = [
        "brand_name",
        "rating_1_count",
        "GmvTotal7",
        "GmvTotal30",
        "GmvTotal90",
        "OrderAcceptedCountTotal7",
        "OrderAcceptedCountTotal30",
        "OrderAcceptedCountTotal90",
        "ItemVarietyCount"
    ]
    
    # Создаем бинарные признаки
    for col in columns_to_check:
        # Формируем имя нового столбца
        new_col_name = f"{col.lower().replace(' ', '_')}_nun"
        
        # Создаем флаг отсутствия значений
        df[new_col_name] = np.where(df[col].isna(), 1, 0)
    
    return df



In [ ]:

df = pd.read_csv('/Users/evgenia/Documents/hack/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_train.csv')
test = pd.read_csv('/Users/evgenia/Documents/hack/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_test.csv')

In [ ]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    df = calculate_stats(df)

    df['russian_ratio_desc'] = df['description'].apply(russian_ratio)
    df['russian_ratio_name'] = df['name_rus'].apply(russian_ratio)
    
    # Мессенджеры
    df['messengers'] = df['description'].apply(find_messenger_mentions)
    df['messengers_2'] = df['name_rus'].apply(find_messenger_mentions)
    df['has_messengers'] = df['messengers'].apply(has_messengers)
    df['has_messengers_2'] = df['messengers_2'].apply(has_messengers)

    # Ссылки
    df = extract_urls(df, "description")
    df['has_urls'] = df['urls'].apply(has_urls)
    
    # Телефоны
    df['rus_phones'] = df['description'].apply(find_phones)
    df['has_rus_phones'] = df['rus_phones'].apply(has_russian_phones)
    
    text_cols = ['description', 'name_rus']
    for col in text_cols:
        df[col] = df[col].apply(safe_clean)
        df[col] = df[col].apply(safe_lemmatize)
    
    # 2.4 Объединение текста
    df['text_all'] = (df['description'] + " " + df['name_rus']).str.strip()
    
    # 2.5 Подозрительные слова
    df['suspect_cnt_name'] = df['name_rus'].apply(count_suspicious)
    df['suspect_cnt_desc'] = df['description'].apply(count_suspicious)
    df['suspect_cnt_total'] = df['suspect_cnt_name'] + df['suspect_cnt_desc']
    
    df = add_activity_features(df)
    df = add_time_features(df)
    df = add_ratios(df)
    df = create_missing_flags(df)

    return df
    
    
df = preprocess_data(df)
test = preprocess_data(test)
